# Data Processing & Feature Engineering

**Owner:** M4 — [@Pranathi-N-47](https://github.com/Pranathi-N-47)  
**Workstream:** data-engineering | **Milestone:** Review 1  

| Section | Output |
|---|---|
| Data Cleaning | `data/processed/features.csv`, `data/processed/features_manhattan.csv`, `data/processed/features_queens.csv` + `cleaning_report.md` |
| Hourly Aggregation & Zero-Fill | `data/processed/hourly_demand.csv` |
| Weather Merge | `hourly_demand.csv` + weather columns |
| Time Features | `hourly_demand.csv` + calendar columns |
| Lag & Rolling Features | `data/processed/features.csv` |

> `data/raw/` is never modified. All outputs go to `data/processed/`.

---
## Setup

In [ ]:
import pandas as pd
import numpy as np
import holidays
from pathlib import Path

ROOT          = Path('../')
RAW           = ROOT / 'data' / 'raw'
PROCESSED     = ROOT / 'data' / 'processed'
PROCESSED.mkdir(parents=True, exist_ok=True)

TRIPS_FILE    = RAW / 'NYC-taxi-demand-Manhattan-Queens.csv'
WEATHER_FILE  = RAW / 'weather_hourly_by_zone.csv'
SUMMARY_FILE  = RAW / 'taxi_collection_summary.csv'

FEATURES_FILE = PROCESSED / 'features.csv'
REPORT_FILE   = PROCESSED / 'cleaning_report.md'

print('Paths OK.')

Paths OK. Processed dir: C:\Users\prana\Desktop\BA\CapStone\nyc-taxi-demand-forecasting\data\processed


---
## Data Cleaning

Four removal rules applied in order:

1. Exact duplicate rows
2. Trips where `dropoff_datetime <= pickup_datetime`
3. Trips lasting more than 6 hours
4. Trips with `DOLocationID` in {264, 265} (Unknown / Outside NYC)

The DST double-hour on 2025-11-02 01:00 is flagged but not removed.

In [22]:
TRIP_COLS = ['taxi_type', 'pickup_datetime', 'dropoff_datetime', 'PULocationID', 'DOLocationID']

print('Loading raw trips ...  (may take a minute for the 3.2 GB file)')
trips = pd.read_csv(
    TRIPS_FILE,
    usecols=TRIP_COLS,
    parse_dates=['pickup_datetime', 'dropoff_datetime'],
    dtype={'taxi_type': 'category', 'PULocationID': 'int16', 'DOLocationID': 'int16'},
)
n_raw = len(trips)
print(f'Raw rows: {n_raw:,}')

Loading raw trips ...  (may take a minute for the 3.2 GB file)


KeyboardInterrupt: 

In [ ]:
# Cross-check expected counts against the collection audit file
summary = pd.read_csv(SUMMARY_FILE)
print('Expected counts from taxi_collection_summary.csv:')
print(f'  Duplicates           : {int(summary["duplicate_rows"].sum()):,}')
print(f'  Bad timestamps       : {int(summary["dropoff_not_after_pickup"].sum()):,}')
print(f'  Trips > 6 h          : {int(summary["longer_than_6h"].sum()):,}')
print(f'  Unknown dropoff zone : {int(summary["dropoff_zone_unknown"].sum()):,}')

In [ ]:
# Rule 1: Exact duplicates
dup_mask = trips.duplicated(subset=TRIP_COLS, keep='first')
n_dup    = dup_mask.sum()
trips    = trips[~dup_mask].copy()
print(f'Removed duplicates    : {n_dup:,}')

In [ ]:
# Rule 2: Dropoff at or before pickup
bad_time_mask = trips['dropoff_datetime'] <= trips['pickup_datetime']
n_bad_time    = bad_time_mask.sum()
trips         = trips[~bad_time_mask].copy()
print(f'Removed bad timestamps: {n_bad_time:,}')

In [ ]:
# Rule 3: Trip duration > 6 hours
duration_h     = (trips['dropoff_datetime'] - trips['pickup_datetime']).dt.total_seconds() / 3600
long_trip_mask = duration_h > 6
n_long         = long_trip_mask.sum()
trips          = trips[~long_trip_mask].copy()
print(f'Removed trips > 6 h   : {n_long:,}')

In [ ]:
# Rule 4: Unknown / Outside NYC dropoff zone
unknown_mask = trips['DOLocationID'].isin([264, 265])
n_unknown    = unknown_mask.sum()
trips        = trips[~unknown_mask].copy()
print(f'Removed unknown DOLoc : {n_unknown:,}')

In [ ]:
# DST note: 2025-11-02 01:00 occurred twice due to clocks falling back
dst_hour  = pd.Timestamp('2025-11-02 01:00:00')
dst_count = ((trips['pickup_datetime'] >= dst_hour) &
             (trips['pickup_datetime'] <  dst_hour + pd.Timedelta(hours=1))).sum()
print(f'DST note: {dst_count:,} trips share pickup_hour 2025-11-02 01:00 (two real clock-hours merged).')
print('  Kept as-is. This hour will show ~2x normal volume in aggregations.')

In [ ]:
# Summary
n_clean   = len(trips)
n_removed = n_raw - n_clean
print(f'Rows raw     : {n_raw:,}')
print(f'Rows removed : {n_removed:,}  ({n_removed/n_raw*100:.2f}%)')
print(f'Rows clean   : {n_clean:,}')



In [ ]:
# Write cleaning_report.md
report = f"""# Cleaning Report

Generated by `scripts/data_processing_engineering.ipynb`.

## Row counts

| Step | Rows removed | Cumulative kept | % of raw |
|---|---|---|---|
| Raw input | -- | {n_raw:,} | 100.00% |
| Rule 1: Exact duplicates | {n_dup:,} | {n_raw - n_dup:,} | {(n_raw - n_dup)/n_raw*100:.2f}% |
| Rule 2: Dropoff <= pickup | {n_bad_time:,} | {n_raw - n_dup - n_bad_time:,} | {(n_raw - n_dup - n_bad_time)/n_raw*100:.2f}% |
| Rule 3: Duration > 6 h | {n_long:,} | {n_raw - n_dup - n_bad_time - n_long:,} | {(n_raw - n_dup - n_bad_time - n_long)/n_raw*100:.2f}% |
| Rule 4: DOLocationID 264/265 | {n_unknown:,} | {n_clean:,} | {n_clean/n_raw*100:.2f}% |

**Total removed:** {n_removed:,} ({n_removed/n_raw*100:.2f}%)  
**Final clean rows:** {n_clean:,}

## DST artefact

On 2025-11-02 the clock fell back at 02:00, making 01:00-01:59 happen twice.  
{dst_count:,} trips carry `pickup_datetime` in that window. They are kept but this
hour shows ~2x normal volume in aggregations.

## Output

`data/processed/features.csv`, `data/processed/features_manhattan.csv`, `data/processed/features_queens.csv`
"""
REPORT_FILE.write_text(report, encoding='utf-8')
print(f'Saved -> {REPORT_FILE}')

---
## Hourly Aggregation & Zero-Fill

A plain `groupby` on trips produces no row for zone-hours with zero pickups.  
The weather table already covers every zone and every hour, so we use it as the  
canonical 138 zones × 4,344 hours grid and left-join trip counts onto it, filling gaps with 0.

In [ ]:
# Derive pickup_hour and count trips per zone per hour
trips['pickup_hour'] = trips['pickup_datetime'].dt.floor('h')

trip_counts = (
    trips.groupby(['PULocationID', 'pickup_hour'], observed=True)
         .size()
         .reset_index(name='trip_count')
)
print(f'Zone-hour pairs with at least one trip: {len(trip_counts):,}')

In [ ]:
# Build the full grid from the weather table
grid = pd.read_csv(
    WEATHER_FILE,
    usecols=['LocationID', 'weather_hour'],
    parse_dates=['weather_hour'],
    dtype={'LocationID': 'int16'},
).rename(columns={'LocationID': 'PULocationID', 'weather_hour': 'pickup_hour'})

n_zones = grid['PULocationID'].nunique()
n_hours = grid['pickup_hour'].nunique()
EXPECTED_ROWS = n_zones * n_hours

print(f'Grid: {n_zones} zones x {n_hours} hours = {EXPECTED_ROWS:,} rows')

In [ ]:
# Left-join trip counts onto the full grid; fill missing hours with 0
demand = grid.merge(trip_counts, on=['PULocationID', 'pickup_hour'], how='left')
demand['trip_count'] = demand['trip_count'].fillna(0).astype('int32')

assert len(demand) == EXPECTED_ROWS, f'Row count mismatch: got {len(demand):,}'
print(f'Row count = {len(demand):,}  PASS')

zero_rows = (demand['trip_count'] == 0).sum()
print(f'Zero-trip rows : {zero_rows:,}  ({zero_rows/len(demand)*100:.1f}%)')
print(f'Non-zero rows  : {len(demand) - zero_rows:,}  ({(len(demand) - zero_rows)/len(demand)*100:.1f}%)')



---
## Weather Merge

Join `apparent_temperature`, `rain`, `snowfall`, `precipitation_probability` from  
`weather_hourly_by_zone.csv` onto the demand grid.  
Because the grid was built from the weather table, every zone-hour is guaranteed  
to match — assert zero nulls after the join.

In [ ]:
weather = pd.read_csv(
    WEATHER_FILE,
    usecols=['LocationID', 'weather_hour',
             'apparent_temperature', 'rain', 'snowfall', 'precipitation_probability'],
    parse_dates=['weather_hour'],
    dtype={
        'LocationID': 'int16',
        'apparent_temperature': 'float32',
        'rain': 'float32',
        'snowfall': 'float32',
        'precipitation_probability': 'int8',
    },
).rename(columns={'LocationID': 'PULocationID', 'weather_hour': 'pickup_hour'})

demand = demand.merge(weather, on=['PULocationID', 'pickup_hour'], how='left')

null_count = demand.isnull().sum().sum()
assert null_count == 0, f'Nulls after weather merge: {null_count}'
print(f'Null count after merge : {null_count}  PASS')
print(f'Zones                  : {demand["PULocationID"].nunique()}')
print(f'Date range             : {demand["pickup_hour"].min()} -> {demand["pickup_hour"].max()}')

print('Columns:', demand.columns.tolist())

---
## Time Features

Derive calendar features from `pickup_hour`:

| Feature | Description |
|---|---|
| `hour` | 0–23 |
| `day_of_week` | 0 (Mon) – 6 (Sun) |
| `month` | 1–12 |
| `is_weekend` | True if Saturday or Sunday |
| `is_holiday` | True if US federal holiday |

In [ ]:
demand['hour']        = demand['pickup_hour'].dt.hour.astype('int8')
demand['day_of_week'] = demand['pickup_hour'].dt.dayofweek.astype('int8')  # 0=Mon, 6=Sun
demand['month']       = demand['pickup_hour'].dt.month.astype('int8')
demand['is_weekend']  = (demand['day_of_week'] >= 5)

us_holidays   = holidays.country_holidays('US', years=[2025, 2026])
holiday_dates = set(us_holidays.keys())
demand['is_holiday'] = demand['pickup_hour'].dt.date.isin(holiday_dates)

# Validation
assert demand['hour'].between(0, 23).all(),       'hour out of range'
assert demand['day_of_week'].between(0, 6).all(), 'day_of_week out of range'
assert demand['month'].isin(range(1, 13)).all(),  'month out of range'

print(f'hour range        : {demand["hour"].min()} - {demand["hour"].max()}  PASS')
print(f'day_of_week range : {demand["day_of_week"].min()} - {demand["day_of_week"].max()}  PASS')
print(f'months present    : {sorted(demand["month"].unique())}  PASS')
print(f'is_holiday = True : {demand["is_holiday"].sum():,} rows')

# Show holidays in the study period
in_range = [(d, n) for d, n in us_holidays.items()
            if pd.Timestamp('2025-09-01') <= pd.Timestamp(d) <= pd.Timestamp('2026-02-28')]
print('\nUS federal holidays in study period:')
for d, name in sorted(in_range):
    print(f'  {d}  {name}')

print(f'\nUpdated -> {DEMAND_FILE}')

---
## Lag & Rolling Features

Computed within each zone, sorted by `pickup_hour`:

| Feature | Description |
|---|---|
| `trip_count_lag_1h` | demand 1 hour ago |
| `trip_count_lag_24h` | demand same hour yesterday |
| `trip_count_lag_168h` | demand same hour last week |
| `trip_count_roll_3h` | 3-hour rolling mean (excludes current row) |
| `trip_count_roll_24h` | 24-hour rolling mean (excludes current row) |

The first 168 rows per zone have no valid 168h lag and are dropped.  
This removes the first 7 days of each zone as training examples (not the underlying trip data).  
The final table `features.csv` is the single input for all models and the dashboard.

In [ ]:
# Sort within each zone -- required to prevent data leakage
demand = demand.sort_values(['PULocationID', 'pickup_hour']).reset_index(drop=True)

grp = demand.groupby('PULocationID', observed=True)['trip_count']

# Point lags
demand['trip_count_lag_1h']   = grp.shift(1)
demand['trip_count_lag_24h']  = grp.shift(24)
demand['trip_count_lag_168h'] = grp.shift(168)

# Rolling means (shift before rolling so the current row is never included)
demand['trip_count_roll_3h'] = (
    grp.transform(lambda x: x.shift(1).rolling(3,  min_periods=3).mean())
).astype('float32')

demand['trip_count_roll_24h'] = (
    grp.transform(lambda x: x.shift(1).rolling(24, min_periods=24).mean())
).astype('float32')

print(f'Rows before dropping warm-up : {len(demand):,}')
print(f'NaN rows                     : {demand.isnull().any(axis=1).sum():,}')

In [ ]:
# Drop warm-up rows (first 168 hours per zone -- NaN from the 168h lag)
n_zones   = demand['PULocationID'].nunique()
features  = demand.dropna().reset_index(drop=True)
n_dropped = len(demand) - len(features)

print(f'Warm-up rows dropped : {n_dropped:,}  (168 x {n_zones} zones = {168 * n_zones:,})')
print(f'Final rows           : {len(features):,}')

# Cast lag columns to int32
for col in ['trip_count_lag_1h', 'trip_count_lag_24h', 'trip_count_lag_168h']:
    features[col] = features[col].astype('int32')

# Final validation
assert features.isnull().sum().sum() == 0, 'Nulls remain in features'
print(f'Null count : 0  PASS')
print(f'Zones      : {features["PULocationID"].nunique()}')
print(f'Date range : {features["pickup_hour"].min()} -> {features["pickup_hour"].max()}')


In [ ]:
# Join borough info from the zones lookup
ZONES_FILE = RAW / 'taxi_zones_manhattan_queens.csv'
zones = pd.read_csv(
    ZONES_FILE,
    usecols=['LocationID', 'Borough']
).rename(columns={'LocationID': 'PULocationID'})

features = features.merge(zones, on='PULocationID', how='left')

# Split by borough
feat_manhattan = features[features['Borough'] == 'Manhattan'].drop(columns='Borough').reset_index(drop=True)
feat_queens    = features[features['Borough'] == 'Queens'].drop(columns='Borough').reset_index(drop=True)
feat_all       = features.drop(columns='Borough').reset_index(drop=True)

# Save all three files
feat_manhattan.to_csv(PROCESSED / 'features_manhattan.csv', index=False)
feat_queens.to_csv(   PROCESSED / 'features_queens.csv',    index=False)
feat_all.to_csv(      FEATURES_FILE,                         index=False)

print(f'features_manhattan.csv : {len(feat_manhattan):,} rows  ({feat_manhattan["PULocationID"].nunique()} zones)')
print(f'features_queens.csv    : {len(feat_queens):,} rows  ({feat_queens["PULocationID"].nunique()} zones)')
print(f'features.csv           : {len(feat_all):,} rows  ({feat_all["PULocationID"].nunique()} zones)')


In [ ]:
# Schema and preview
print('Final schema:')
print(feat_all.dtypes.to_string())
print()
feat_all.head(3)
